# PCB model MLOps: drift detection, automated retraining, redeployment and version comparison

This notebook extends the PCB computer vision demo (notebooks `01` to `05`) with an end-to-end, automatable MLOps loop built on Databricks and MLflow:

| Stage | What happens | Databricks / MLflow capability |
|---|---|---|
| 1. Incoming data | New PCB images arrive (simulated drifted batch and/or real endpoint traffic) | Delta tables, AI Gateway inference table |
| 2. Scoring & profiling | The `@production` model scores the images, image statistics are computed and appended to an inference log | MLflow model registry in Unity Catalog, Delta |
| 3. Drift detection | Statistical tests (PSI, KS) on image features, prediction mix, confidence and accuracy | MLflow tracking (metrics, tables, figures), Data Quality Monitoring |
| 4. Retraining | If drift is detected, the model is fine-tuned on original + new labelled data and registered as `@challenger` | MLflow transformers flavor, UC model versions, dataset lineage |
| 5. Comparison | `@challenger` vs `@production` on the same held-out set (original vs drifted segments) | MLflow runs comparison, nested runs, version tags |
| 6. Redeployment | If the promotion gate passes: alias swap for batch, new real-time version on the serving endpoint (full or canary) | UC aliases, Model Serving, inference tables |
| 7. Automation | A Lakeflow Job chains the steps with condition tasks | Lakeflow Jobs, task values |

**Compatibility with the existing notebooks:** nothing in `01`-`05`, `config` or `_resources` is changed. This notebook reuses the same catalog/schema, the same registered model (`dbdemos_pcb_classification`), the same `production` alias, the same MLflow experiment and the same serving endpoint. Notebook `03` keeps loading `models:/...@production` and automatically picks up a promoted model.

Run the whole notebook interactively with `step = all`, or let the Lakeflow Job created at the end run each step as a separate task. A detailed explanation of the full process is at the end of the notebook.

In [0]:
%pip install databricks-sdk mlflow==3.14.0 "transformers>=4.46,<5" datasets accelerate evaluate torch torchvision "huggingface-hub==0.36.2" scipy scikit-learn matplotlib
dbutils.library.restartPython()

In [0]:
%run ./_resources/00-init $reset_all_data=false

## Configuration

All behaviour is controlled by widgets so the same notebook can be run interactively or as individual tasks of a Lakeflow Job:

* `step`: `all` runs the full loop; `drift_check`, `retrain`, `evaluate`, `deploy` run a single stage (used by the job); `rollback` restores the previous production version.
* `simulate_drift`: creates a new batch of altered images (darker, blurred, noisy, rotated) to demonstrate drift. Set to `false` once real images flow into `pcb_incoming_images`.
* `include_endpoint_traffic`: also profiles real requests captured by the serving endpoint inference table.
* Drift thresholds (`psi_threshold`, `ks_alpha`, `min_drifted_features`, `confidence_drop`, `f1_drop`) decide when drift is significant.
* Promotion gate (`min_f1_gain`, `max_f1_regression`, `auto_promote`) decides when the retrained model replaces production.
* `rt_rollout`: `full` switches all endpoint traffic to the new version, `canary` sends `canary_pct`% to it.

In [0]:
dbutils.widgets.dropdown("step", "all", ["all", "drift_check", "retrain", "evaluate", "deploy", "rollback"], "Step to run")
dbutils.widgets.dropdown("simulate_drift", "true", ["true", "false"], "Simulate a drifted batch")
dbutils.widgets.dropdown("include_endpoint_traffic", "false", ["true", "false"], "Profile endpoint traffic")
dbutils.widgets.text("psi_threshold", "0.2", "PSI threshold")
dbutils.widgets.text("ks_alpha", "0.01", "KS p-value threshold")
dbutils.widgets.text("min_drifted_features", "2", "Min drifted features")
dbutils.widgets.text("confidence_drop", "0.05", "Max mean confidence drop")
dbutils.widgets.text("f1_drop", "0.05", "Max F1 drop (labelled data)")
dbutils.widgets.dropdown("force_retrain", "false", ["true", "false"], "Force retraining")
dbutils.widgets.text("num_train_epochs", "5", "Retraining epochs")
dbutils.widgets.text("min_f1_gain", "0.0", "Min F1 gain on drifted data")
dbutils.widgets.text("max_f1_regression", "0.02", "Max F1 loss on original data")
dbutils.widgets.dropdown("auto_promote", "true", ["true", "false"], "Auto promote if gate passes")
dbutils.widgets.dropdown("rt_rollout", "full", ["full", "canary"], "Real-time rollout")
dbutils.widgets.text("canary_pct", "10", "Canary traffic %")
dbutils.widgets.dropdown("enable_monitoring", "true", ["true", "false"], "Data quality monitoring")
dbutils.widgets.dropdown("create_job", "true", ["true", "false"], "Create/update automation job")

STEP = dbutils.widgets.get("step")
SIMULATE_DRIFT = dbutils.widgets.get("simulate_drift") == "true"
INCLUDE_ENDPOINT_TRAFFIC = dbutils.widgets.get("include_endpoint_traffic") == "true"
PSI_THRESHOLD = float(dbutils.widgets.get("psi_threshold"))
KS_ALPHA = float(dbutils.widgets.get("ks_alpha"))
MIN_DRIFTED_FEATURES = int(dbutils.widgets.get("min_drifted_features"))
CONFIDENCE_DROP = float(dbutils.widgets.get("confidence_drop"))
F1_DROP = float(dbutils.widgets.get("f1_drop"))
FORCE_RETRAIN = dbutils.widgets.get("force_retrain") == "true"
NUM_TRAIN_EPOCHS = int(dbutils.widgets.get("num_train_epochs"))
MIN_F1_GAIN = float(dbutils.widgets.get("min_f1_gain"))
MAX_F1_REGRESSION = float(dbutils.widgets.get("max_f1_regression"))
AUTO_PROMOTE = dbutils.widgets.get("auto_promote") == "true"
RT_ROLLOUT = dbutils.widgets.get("rt_rollout")
CANARY_PCT = int(dbutils.widgets.get("canary_pct"))
ENABLE_MONITORING = dbutils.widgets.get("enable_monitoring") == "true"
CREATE_JOB = dbutils.widgets.get("create_job") == "true"

def should_run(stage):
  return STEP in ("all", stage)

# Same names as notebooks 02/03 so nothing else has to change
MODEL_NAME = f"{catalog}.{db}.dbdemos_pcb_classification"
SERVING_ENDPOINT_NAME = "dbdemos_pcb_classification_endpoint"
PAYLOAD_TABLE = f"{catalog}.{db}.{SERVING_ENDPOINT_NAME}_payload"   # inference table created by notebook 03
TRAINING_TABLE = f"{catalog}.{db}.training_dataset_augmented"
PROD_ALIAS, CHALLENGER_ALIAS, PREVIOUS_ALIAS = "production", "challenger", "previous"   # UC aliases are case-insensitive: 'Production' == 'production'

# New tables owned by this notebook only
INCOMING_TABLE = f"{catalog}.{db}.pcb_incoming_images"     # new images to check for drift (labels optional)
BASELINE_TABLE = f"{catalog}.{db}.pcb_drift_baseline"      # reference profile of the training data, scored by the production model
INFERENCE_LOG = f"{catalog}.{db}.pcb_inference_log"        # scored + profiled production data (monitored table)
EVENTS_TABLE = f"{catalog}.{db}.pcb_mlops_events"          # audit trail of every decision taken by this notebook

IMAGE_FEATURES = ["brightness", "contrast", "sharpness", "entropy", "red_mean", "green_mean", "blue_mean"]

import mlflow
mlflow.set_registry_uri("databricks-uc")
DBDemos.init_experiment_for_batch("computer-vision-dl", "pcb")   # same experiment as notebooks 02/03
print(f"step={STEP} | model={MODEL_NAME} | endpoint={SERVING_ENDPOINT_NAME}")

In [0]:
import io, json, time, uuid, datetime
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter
from mlflow import MlflowClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

client = MlflowClient()
DEVICE = 0 if torch.cuda.is_available() else -1   # transformers pipeline device: GPU 0 or CPU

# ---------- Unity Catalog model registry ----------
def get_version_by_alias(alias):
  """Return the ModelVersion behind an alias, or None if the alias doesn't exist."""
  try:
    return client.get_model_version_by_alias(MODEL_NAME, alias)
  except Exception:
    return None

def load_pipeline(version):
  """Load a registered transformers version as a HF pipeline (same loading code as notebook 03)."""
  return mlflow.transformers.load_model(f"models:/{MODEL_NAME}/{version}", return_type="pipeline", device=DEVICE)

# ---------- Audit trail of decisions (also used to hand state between job tasks) ----------
EVENT_SCHEMA = "event_ts timestamp, stage string, model_version string, decision string, details string, mlflow_run_id string"

def log_event(stage, decision, model_version=None, run_id=None, **details):
  row = [(datetime.datetime.utcnow(), stage, None if model_version is None else str(model_version), decision, json.dumps(details, default=str), run_id)]
  spark.createDataFrame(row, EVENT_SCHEMA).write.mode("append").saveAsTable(EVENTS_TABLE)

def latest_event(stage, model_version=None):
  if not spark.catalog.tableExists(EVENTS_TABLE):
    return None
  df = spark.table(EVENTS_TABLE).filter(F.col("stage") == stage)
  if model_version is not None:
    df = df.filter(F.col("model_version") == str(model_version))
  rows = df.orderBy(F.col("event_ts").desc()).limit(1).collect()
  return rows[0].asDict() if rows else None

def set_task_value(key, value):
  """Expose a value to downstream job tasks (used by condition tasks). No-op when run interactively."""
  try:
    dbutils.jobs.taskValues.set(key=key, value=value)
  except Exception:
    pass

# ---------- Image profiling (features used for drift detection) ----------
def image_features(content):
  img = Image.open(io.BytesIO(content)).convert("RGB")
  arr = np.asarray(img, dtype=np.float32)
  gray = arr.mean(axis=2)
  edges = np.asarray(img.convert("L").filter(ImageFilter.FIND_EDGES), dtype=np.float32)
  hist = np.histogram(gray, bins=64, range=(0, 255))[0].astype(np.float64)
  p = hist[hist > 0] / hist.sum()
  return {"brightness": float(gray.mean()), "contrast": float(gray.std()), "sharpness": float(edges.var()),
          "entropy": float(-(p * np.log2(p)).sum()), "red_mean": float(arr[..., 0].mean()),
          "green_mean": float(arr[..., 1].mean()), "blue_mean": float(arr[..., 2].mean())}

# ---------- Scoring ----------
def predict_contents(pipe, contents, batch_size=16):
  """Run the HF pipeline on raw image bytes. Returns predicted label, its score, P(damaged) and latency per image."""
  images = [Image.open(io.BytesIO(b)).convert("RGB") for b in contents]
  start = time.time()
  # No batch_size: pipelines logged by notebook 02 use the image processor as 'tokenizer', which breaks HF batching
  outputs = pipe(images, top_k=2)
  latency_ms = (time.time() - start) * 1000 / max(len(images), 1)
  best = [max(o, key=lambda x: x["score"]) for o in outputs]
  p_damaged = [next((x["score"] for x in o if x["label"] == "damaged"), 0.0) for o in outputs]
  return pd.DataFrame({"prediction": [b["label"] for b in best], "score": [float(b["score"]) for b in best],
                       "p_damaged": [float(p) for p in p_damaged]}), latency_ms

def score_and_profile(pdf, pipe, model_version):
  """pdf needs columns image_id, content, label, source. Returns features + predictions (inference log schema)."""
  preds, _ = predict_contents(pipe, pdf["content"].tolist())
  feats = pd.DataFrame([image_features(b) for b in pdf["content"]])
  out = pd.concat([pdf[["image_id", "source", "label"]].reset_index(drop=True), feats, preds], axis=1)
  out["model_version"] = str(model_version)
  out["scored_at"] = pd.Timestamp.utcnow().tz_localize(None)
  return out

INFERENCE_LOG_SCHEMA = ("image_id string, source string, label string, " + ", ".join(f"{f} double" for f in IMAGE_FEATURES) +
                        ", prediction string, score double, p_damaged double, model_version string, scored_at timestamp")

# ---------- Drift statistics ----------
def psi(expected, actual, bins=10):
  """Population Stability Index on quantile bins of the reference distribution. <0.1 stable, 0.1-0.2 moderate, >0.2 significant."""
  expected, actual = np.asarray(expected, float), np.asarray(actual, float)
  edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
  edges[0], edges[-1] = -np.inf, np.inf
  e = np.clip(np.histogram(expected, edges)[0] / len(expected), 1e-6, None)
  a = np.clip(np.histogram(actual, edges)[0] / len(actual), 1e-6, None)
  return float(np.sum((a - e) * np.log(a / e)))

def categorical_psi(expected, actual):
  cats = sorted(set(expected) | set(actual))
  e = np.clip(np.array([np.mean(np.asarray(expected) == c) for c in cats]), 1e-6, None)
  a = np.clip(np.array([np.mean(np.asarray(actual) == c) for c in cats]), 1e-6, None)
  return float(np.sum((a - e) * np.log(a / e)))

# ---------- Classification metrics ('damaged' is the positive class) ----------
def clf_metrics(y_true, y_pred, p_damaged):
  yt = (np.asarray(y_true) == "damaged").astype(int)
  yp = (np.asarray(y_pred) == "damaged").astype(int)
  m = {"accuracy": accuracy_score(yt, yp), "precision": precision_score(yt, yp, zero_division=0),
       "recall": recall_score(yt, yp, zero_division=0), "f1": f1_score(yt, yp, zero_division=0)}
  m["roc_auc"] = roc_auc_score(yt, p_damaged) if len(set(yt)) > 1 else float("nan")
  return {k: float(v) for k, v in m.items()}

## Stage 1 - Incoming data

In production, new PCB images land continuously (factory cameras, the real-time endpoint). Drift typically comes from changes in the capture conditions: new lighting, a dirty or defocused lens, sensor noise, a different camera angle.

All new images are collected in `pcb_incoming_images` (one `batch_id` per arrival). Labels are optional; when the QA team labels images, they can be used both to measure accuracy drift and to retrain.

To demonstrate the loop, `simulate_drift = true` creates a new batch by taking existing PCB images and applying realistic capture changes (darker lighting, lower contrast, blur, sensor noise, small rotation). Labels are kept, so the batch can also be used for retraining. Each image is assigned deterministically to a `train` (80%) or `holdout` (20%) split, so the evaluation data is never used for retraining.

With `include_endpoint_traffic = true`, the requests captured by the serving endpoint inference table (created by notebook `03`) are also profiled (without labels).

In [0]:
from PIL import ImageEnhance

INCOMING_SCHEMA = "image_id string, filename string, content binary, label string, source string, split string, batch_id string, ingested_at timestamp"

def simulate_capture_drift(content, rng):
  """Simulate a change in the factory capture conditions."""
  img = Image.open(io.BytesIO(content)).convert("RGB")
  img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.45, 0.65))      # darker lighting
  img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.6, 0.8))          # washed-out contrast
  img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(1.0, 2.0))) # defocused lens
  arr = np.asarray(img, dtype=np.float32) + rng.normal(0, 12, (img.size[1], img.size[0], 3))  # sensor noise
  img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)).rotate(rng.uniform(-8, 8))    # camera angle
  out = io.BytesIO(); img.save(out, format="JPEG")
  return out.getvalue()

if should_run("drift_check"):
  batch_id = datetime.datetime.utcnow().strftime("%Y%m%d%H%M%S")
  new_batches = []

  if SIMULATE_DRIFT:
    rng = np.random.default_rng()
    src = (spark.table(TRAINING_TABLE).filter("label = 'damaged'").orderBy(F.rand()).limit(100)
           .unionByName(spark.table(TRAINING_TABLE).filter("label = 'normal'").orderBy(F.rand()).limit(200))
           .select("filename", "content", "label").toPandas())
    src["content"] = src["content"].apply(lambda b: simulate_capture_drift(bytes(b), rng))
    src["image_id"] = [str(uuid.uuid4()) for _ in range(len(src))]
    src["source"] = "simulated_drift"
    new_batches.append(spark.createDataFrame(src[["image_id", "filename", "content", "label", "source"]]))

  if INCLUDE_ENDPOINT_TRAFFIC and spark.catalog.tableExists(PAYLOAD_TABLE):
    # Requests sent by the RT wrapper are {"dataframe_records": [{"data": <base64 image>}]}
    req = F.from_json("request", "struct<dataframe_records: array<struct<data: string>>>")
    endpoint_df = (spark.table(PAYLOAD_TABLE)
                   .filter((F.col("status_code") == 200) & (F.col("request_time") >= F.expr("current_timestamp() - INTERVAL 7 DAYS")))
                   .select(F.explode(req.dataframe_records).alias("r"), "databricks_request_id")
                   .select(F.expr("uuid()").alias("image_id"), F.col("databricks_request_id").alias("filename"),
                           F.unbase64("r.data").alias("content"), F.lit(None).cast("string").alias("label"),
                           F.lit("endpoint").alias("source")))
    new_batches.append(endpoint_df)

  if new_batches:
    batch_df = new_batches[0]
    for b in new_batches[1:]:
      batch_df = batch_df.unionByName(b)
    (batch_df
      .withColumn("split", F.when(F.abs(F.xxhash64("image_id")) % 10 < 8, "train").otherwise("holdout"))
      .withColumn("batch_id", F.lit(batch_id))
      .withColumn("ingested_at", F.current_timestamp())
      .selectExpr(*[c.split(" ")[0] for c in INCOMING_SCHEMA.split(", ")])
      .write.mode("append").option("mergeSchema", "true").saveAsTable(INCOMING_TABLE))

  # The current window is the latest batch that arrived (generated now, or written by an upstream ingestion process)
  assert spark.catalog.tableExists(INCOMING_TABLE), f"No incoming data yet: enable simulate_drift or load images into {INCOMING_TABLE}"
  current_batch_id = spark.table(INCOMING_TABLE).agg(F.max("batch_id")).first()[0]
  current_df = spark.table(INCOMING_TABLE).filter(F.col("batch_id") == current_batch_id)
  print(f"Current batch {current_batch_id}: {current_df.count()} images")
  display(current_df.groupBy("source", "label", "split").count())

## Stage 2 - Score and profile with the production model

The model behind the `@production` alias (the same one used by notebook `03`) scores both:

* **The baseline**: a fixed sample of the training table `training_dataset_augmented`. It represents what the model was trained on. It is stored in `pcb_drift_baseline` and recomputed only when the production version changes.
* **The current batch**: the newly arrived images. Results are appended to `pcb_inference_log`.

For each image we keep simple, explainable image statistics (brightness, contrast, sharpness, entropy, colour channel means) together with the predicted label, its confidence and `P(damaged)`. Raw pixels are high dimensional, so these statistics are a cheap and interpretable signal of capture changes. The predictions themselves are a signal of *concept* or *prediction* drift.

In [0]:
if should_run("drift_check"):
  champion = get_version_by_alias(PROD_ALIAS)
  assert champion is not None, f"No '{PROD_ALIAS}' alias on {MODEL_NAME}: run notebook 02 first"
  print(f"Production model: {MODEL_NAME} version {champion.version}")
  champion_pipe = load_pipeline(champion.version)

  # Baseline: recompute only if missing or built with another production version
  rebuild = (not spark.catalog.tableExists(BASELINE_TABLE)) or \
            spark.table(BASELINE_TABLE).filter(F.col("model_version") != str(champion.version)).limit(1).count() > 0
  if rebuild:
    base_pdf = (spark.table(TRAINING_TABLE).sample(fraction=1.0, seed=42).limit(400)
                .select(F.col("filename").alias("image_id"), "content", "label").toPandas())
    base_pdf["source"] = "training_baseline"
    base_scored = score_and_profile(base_pdf, champion_pipe, champion.version)
    (spark.createDataFrame(base_scored, INFERENCE_LOG_SCHEMA).withColumn("batch_id", F.lit("baseline")).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(BASELINE_TABLE))
    print(f"Baseline rebuilt with {len(base_scored)} images")
  baseline_pdf = spark.table(BASELINE_TABLE).toPandas()

  # Current batch
  cur_pdf = current_df.select("image_id", "content", "label", "source").toPandas()
  current_scored = score_and_profile(cur_pdf, champion_pipe, champion.version)
  (spark.createDataFrame(current_scored, INFERENCE_LOG_SCHEMA).withColumn("batch_id", F.lit(current_batch_id))
     .write.mode("append").option("mergeSchema", "true").saveAsTable(INFERENCE_LOG))
  spark.sql(f"ALTER TABLE {INFERENCE_LOG} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")  # recommended for monitoring

  display(pd.concat([baseline_pdf.assign(window="baseline"), current_scored.assign(window="current")])
            .groupby("window")[IMAGE_FEATURES + ["score", "p_damaged"]].mean().reset_index())

## Stage 3 - Drift detection, tracked in MLflow

Three kinds of drift are tested between the baseline and the current batch:

| Type | Test | Drift when |
|---|---|---|
| **Data drift** (input images) | PSI and two-sample Kolmogorov-Smirnov test on each image feature | at least `min_drifted_features` features have `PSI > psi_threshold` **and** `KS p-value < ks_alpha` |
| **Prediction drift** | PSI of the predicted-label mix, drop in mean confidence | label PSI > `psi_threshold` or confidence drop > `confidence_drop` |
| **Performance drift** (only if labels exist) | F1 on labelled current images vs baseline | F1 drop > `f1_drop` |

Everything is logged to an MLflow run named `drift_check` in the demo experiment: one metric per test, the full drift report as an MLflow table, histogram figures of baseline vs current, the incoming dataset as a lineage input, and the final decision as a tag. The decision is also written to `pcb_mlops_events` and published as the task value `drift_detected` for the job.

In [0]:
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

if should_run("drift_check"):
  report = []
  for f in IMAGE_FEATURES:
    ks = ks_2samp(baseline_pdf[f], current_scored[f])
    p = psi(baseline_pdf[f], current_scored[f])
    report.append({"test": f"feature:{f}", "psi": p, "ks_stat": float(ks.statistic), "ks_pvalue": float(ks.pvalue),
                   "baseline_mean": float(baseline_pdf[f].mean()), "current_mean": float(current_scored[f].mean()),
                   "drifted": bool(p > PSI_THRESHOLD and ks.pvalue < KS_ALPHA)})
  drift_report = pd.DataFrame(report)
  n_drifted = int(drift_report["drifted"].sum())

  label_psi = categorical_psi(baseline_pdf["prediction"], current_scored["prediction"])
  conf_drop = float(baseline_pdf["score"].mean() - current_scored["score"].mean())
  labelled = current_scored[current_scored["label"].notna()]
  base_f1 = clf_metrics(baseline_pdf["label"], baseline_pdf["prediction"], baseline_pdf["p_damaged"])["f1"]
  cur_f1 = clf_metrics(labelled["label"], labelled["prediction"], labelled["p_damaged"])["f1"] if len(labelled) else None
  f1_drop_value = None if cur_f1 is None else base_f1 - cur_f1

  reasons = []
  if n_drifted >= MIN_DRIFTED_FEATURES: reasons.append(f"data drift on {n_drifted} features")
  if label_psi > PSI_THRESHOLD: reasons.append(f"prediction mix PSI {label_psi:.3f}")
  if conf_drop > CONFIDENCE_DROP: reasons.append(f"confidence drop {conf_drop:.3f}")
  if f1_drop_value is not None and f1_drop_value > F1_DROP: reasons.append(f"F1 drop {f1_drop_value:.3f}")
  drift_detected = len(reasons) > 0

  with mlflow.start_run(run_name="drift_check") as drift_run:
    mlflow.set_tags({"dbdemos": "pcb_classification", "mlops_stage": "drift_check", "champion_version": champion.version,
                     "batch_id": current_batch_id, "drift_detected": str(drift_detected), "drift_reasons": "; ".join(reasons) or "none"})
    mlflow.log_params({"psi_threshold": PSI_THRESHOLD, "ks_alpha": KS_ALPHA, "min_drifted_features": MIN_DRIFTED_FEATURES,
                       "confidence_drop": CONFIDENCE_DROP, "f1_drop": F1_DROP, "n_baseline": len(baseline_pdf), "n_current": len(current_scored)})
    for r in report:
      name = r["test"].split(":")[1]
      mlflow.log_metrics({f"psi_{name}": r["psi"], f"ks_pvalue_{name}": r["ks_pvalue"]})
    mlflow.log_metrics({"n_drifted_features": n_drifted, "prediction_psi": label_psi, "confidence_drop": conf_drop,
                        "baseline_f1": base_f1, "drift_detected": int(drift_detected)})
    if cur_f1 is not None:
      mlflow.log_metrics({"current_f1": cur_f1, "f1_drop": f1_drop_value})
    mlflow.log_table(drift_report, "drift_report.json")
    mlflow.log_input(mlflow.data.load_delta(table_name=INFERENCE_LOG), context="drift-current")
    mlflow.log_input(mlflow.data.load_delta(table_name=BASELINE_TABLE), context="drift-baseline")

    fig, axes = plt.subplots(1, len(IMAGE_FEATURES), figsize=(4 * len(IMAGE_FEATURES), 3))
    for ax, f in zip(axes, IMAGE_FEATURES):
      ax.hist(baseline_pdf[f], bins=30, alpha=0.6, density=True, label="baseline")
      ax.hist(current_scored[f], bins=30, alpha=0.6, density=True, label="current")
      ax.set_title(f); ax.legend()
    plt.tight_layout(); mlflow.log_figure(fig, "drift_histograms.png"); plt.show()

  log_event("drift_check", "drift" if drift_detected else "no_drift", champion.version, drift_run.info.run_id,
            batch_id=current_batch_id, reasons=reasons, n_drifted_features=n_drifted, prediction_psi=label_psi,
            confidence_drop=conf_drop, current_f1=cur_f1, baseline_f1=base_f1)
  set_task_value("drift_detected", str(drift_detected).lower())
  display(drift_report)
  print(f"Drift detected: {drift_detected} -> {reasons or 'model inputs and outputs are stable'}")

### Continuous monitoring with Databricks Data Quality Monitoring

The MLflow drift check above gives a fast go/no-go decision per batch. For long-term observability, `pcb_inference_log` is also registered as an **inference profile monitor** (Data Quality Monitoring, formerly Lakehouse Monitoring):

* problem type `classification`, prediction column `prediction`, label column `label`, model id column `model_version`, timestamp `scored_at`, daily granularity;
* baseline table `pcb_drift_baseline`, so drift is computed both versus the previous time window and versus the training data;
* the monitor writes `pcb_inference_log_profile_metrics` and `pcb_inference_log_drift_metrics` (KS, chi-square, JS distance, PSI, Wasserstein, accuracy/F1 per model version and window) and generates a dashboard.

The monitor is created once and refreshed on every run. Refreshes are asynchronous; the metric tables are updated after a few minutes. You can also add a SQL alert on `pcb_inference_log_drift_metrics`.

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

if should_run("drift_check") and ENABLE_MONITORING:
  from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType
  try:
    try:
      w.quality_monitors.get(table_name=INFERENCE_LOG)
      monitor_exists = True
    except Exception:
      monitor_exists = False
    if not monitor_exists:
      user = w.current_user.me().user_name
      w.quality_monitors.create(
        table_name=INFERENCE_LOG,
        assets_dir=f"/Workspace/Users/{user}/databricks_lakehouse_monitoring/{INFERENCE_LOG}",
        output_schema_name=f"{catalog}.{db}",
        baseline_table_name=BASELINE_TABLE,
        inference_log=MonitorInferenceLog(
          problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
          prediction_col="prediction", label_col="label", model_id_col="model_version",
          timestamp_col="scored_at", granularities=["1 day"]),
        slicing_exprs=["source"])
      print(f"Monitor created on {INFERENCE_LOG} (first refresh starts automatically)")
    else:
      refresh = w.quality_monitors.run_refresh(table_name=INFERENCE_LOG)
      print(f"Monitor refresh {refresh.refresh_id} started on {INFERENCE_LOG}")
    drift_metrics_table = f"{INFERENCE_LOG}_drift_metrics"
    if spark.catalog.tableExists(drift_metrics_table):
      display(spark.table(drift_metrics_table)
              .filter("column_name != ':table'")
              .select("window", "drift_type", "column_name", "slice_key", "slice_value", "ks_test", "js_distance", "population_stability_index")
              .orderBy(F.col("window.start").desc()).limit(50))
  except Exception as e:
    # Monitoring is an observability add-on: never block the retraining loop on it
    print(f"Data Quality Monitoring not configured: {e}")

## Stage 4 - Automated retraining

Runs only if the latest drift check found drift (or `force_retrain = true`).

* **Training data**: the original `training_dataset_augmented` table **plus** the labelled `train` split of `pcb_incoming_images`, so the model learns the new capture conditions without forgetting the original ones. The `holdout` split is never used for training.
* **Training recipe**: same as notebook `02` (Hugging Face ViT fine-tuning, same transformations, F1 as selection metric, early stopping). On GPU compute the larger `google/vit-base-patch16-224` is used, on CPU `WinKawaks/vit-tiny-patch16-224`.
* **MLflow**: a new run `hugging_face_retrain` (the original `hugging_face` runs are untouched), tagged with the drift run that triggered it and the compute type. Both training tables are logged as dataset inputs for Unity Catalog lineage.
* **Registry**: the model is registered as a **new version** of `dbdemos_pcb_classification`, tagged `model_type=transformers`, and receives the `@challenger` alias. `@production` is **not** touched at this point, so batch inference (notebook `03`) and the endpoint keep using the current model.

Why not call notebook `02`? It moves `@production` immediately after training, which would skip the comparison gate below.

In [0]:
def retrain_model(num_epochs, drift_run_id=None):
  from datasets import Dataset
  from transformers import (AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer,
                            EarlyStoppingCallback, pipeline)
  from torchvision.transforms import Compose, Normalize, RandomResizedCrop, Resize, ToTensor, Lambda, ToPILImage
  from mlflow.models.signature import infer_signature
  import evaluate

  # 1. Training data: original table + labelled 'train' split of the incoming images
  cols = ["content", "label"]
  train_sdf = spark.table(TRAINING_TABLE).select(*cols).withColumn("origin", F.lit("original"))
  if spark.catalog.tableExists(INCOMING_TABLE):
    new_sdf = (spark.table(INCOMING_TABLE).filter("split = 'train' AND label IS NOT NULL")
               .select(*cols).withColumn("origin", F.lit("incoming")))
    train_sdf = train_sdf.unionByName(new_sdf)
  pdf = train_sdf.toPandas()
  print(pdf.groupby(["origin", "label"]).size())
  dataset = Dataset.from_pandas(pdf).rename_column("content", "image")
  splits = dataset.train_test_split(test_size=0.2, seed=42)
  train_ds, val_ds = splits["train"], splits["test"]

  # 2. Same model / preprocessing recipe as notebook 02
  model_checkpoint = "google/vit-base-patch16-224" if torch.cuda.is_available() else "WinKawaks/vit-tiny-patch16-224"
  model_def = AutoImageProcessor.from_pretrained(model_checkpoint)
  size = (model_def.size["height"], model_def.size["width"])
  normalize = Normalize(mean=model_def.image_mean, std=model_def.image_std)
  byte_to_pil = Lambda(lambda b: Image.open(io.BytesIO(b)).convert("RGB"))
  train_tf = Compose([byte_to_pil, RandomResizedCrop(size), ToTensor(), normalize])
  val_tf = Compose([byte_to_pil, Resize(size), ToTensor(), normalize])
  train_ds.set_transform(lambda b: {**b, "image": [train_tf(i) for i in b["image"]]})
  val_ds.set_transform(lambda b: {**b, "image": [val_tf(i) for i in b["image"]]})

  labels = sorted(set(dataset["label"]))
  label2id = {l: i for i, l in enumerate(labels)}
  id2label = {i: l for l, i in label2id.items()}
  model = AutoModelForImageClassification.from_pretrained(model_checkpoint, label2id=label2id, id2label=id2label,
                                                          ignore_mismatched_sizes=True)
  f1_metric = evaluate.load("f1")
  def compute_metrics(eval_pred):
    return f1_metric.compute(predictions=np.argmax(eval_pred.predictions, axis=1), references=eval_pred.label_ids,
                             pos_label=label2id["damaged"])
  def collate_fn(examples):
    return {"pixel_values": torch.stack([e["image"] for e in examples]),
            "labels": torch.tensor([label2id[e["label"]] for e in examples])}

  args = TrainingArguments(f"/tmp/huggingface/pcb/{model_checkpoint.split('/')[-1]}-retrain",
                           remove_unused_columns=False, eval_strategy="epoch", save_strategy="epoch",
                           learning_rate=5e-5, per_device_train_batch_size=32, per_device_eval_batch_size=32,
                           use_cpu=not torch.cuda.is_available(), num_train_epochs=num_epochs, warmup_ratio=0.1,
                           logging_steps=10, load_best_model_at_end=True, metric_for_best_model="f1",
                           save_total_limit=2, push_to_hub=False, report_to=["mlflow"])

  with mlflow.start_run(run_name="hugging_face_retrain") as run:
    mlflow.set_tags({"dbdemos": "pcb_classification", "mlops_stage": "retrain", "trigger": "drift" if drift_run_id else "manual",
                     "drift_run_id": drift_run_id or "", "compute": "gpu" if torch.cuda.is_available() else "cpu",
                     "base_checkpoint": model_checkpoint})
    trainer = Trainer(model, args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=model_def,
                      compute_metrics=compute_metrics, data_collator=collate_fn,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    train_results = trainer.train()
    classifier = pipeline("image-classification", model=trainer.state.best_model_checkpoint,
                          image_processor=model_def, device=DEVICE)

    img = ToPILImage()(val_ds[0]["image"])
    signature = infer_signature(model_input=np.array(img), model_output=pd.DataFrame(classifier(img)))
    model_info = mlflow.transformers.log_model(name="model", transformers_model=classifier,
                                               pip_requirements=mlflow.transformers.get_default_pip_requirements(model),
                                               signature=signature)
    mlflow.log_metrics(train_results.metrics)
    mlflow.log_input(mlflow.data.load_delta(table_name=TRAINING_TABLE), context="Training-Input")
    if spark.catalog.tableExists(INCOMING_TABLE):
      mlflow.log_input(mlflow.data.load_delta(table_name=INCOMING_TABLE), context="Training-Input-Incoming")

  # 3. Register as a new version with the @challenger alias (production untouched)
  mv = mlflow.register_model(model_info.model_uri, MODEL_NAME)
  for k, v in {"model_type": "transformers", "trigger": "drift" if drift_run_id else "manual", "training_run_id": run.info.run_id,
               "drift_run_id": drift_run_id or "", "base_checkpoint": model_checkpoint}.items():
    client.set_model_version_tag(MODEL_NAME, mv.version, k, v)
  client.set_registered_model_alias(MODEL_NAME, CHALLENGER_ALIAS, mv.version)
  return mv, run.info.run_id

if should_run("retrain"):
  last_drift = latest_event("drift_check")
  if FORCE_RETRAIN or (last_drift is not None and last_drift["decision"] == "drift"):
    challenger_mv, retrain_run_id = retrain_model(NUM_TRAIN_EPOCHS, last_drift["mlflow_run_id"] if last_drift else None)
    log_event("retrain", "registered_challenger", challenger_mv.version, retrain_run_id,
              drift_run_id=last_drift["mlflow_run_id"] if last_drift else None)
    set_task_value("challenger_version", str(challenger_mv.version))
    print(f"Registered {MODEL_NAME} version {challenger_mv.version} as @{CHALLENGER_ALIAS}")
  else:
    print("No drift detected in the latest check: retraining skipped (set force_retrain=true to override)")

## Stage 5 - Compare the new version (`@challenger`) with the current one (`@production`)

Both versions are scored on **exactly the same evaluation set**, split into two segments:

* `drifted`: the labelled `holdout` split of `pcb_incoming_images` (never used for training) - does the new model handle the new conditions?
* `original`: a fixed sample of the original training table - did the new model forget the original conditions? (In this demo the champion has seen part of these images during training, so this segment is optimistic for the champion. In production, keep a frozen labelled holdout table.)

Metrics per model and segment: accuracy, precision, recall, F1, ROC AUC (`damaged` is the positive class), confusion matrix, latency per image and number of parameters.

In MLflow, a parent run `champion_vs_challenger` holds the comparison table, the deltas and the decision, with one nested run per model version (`eval_v<version>`). Open the experiment, select the two nested runs and click **Compare** to view them side by side. The metrics are also linked to each MLflow 3 logged model, and the result is written as tags on the challenger model version in Unity Catalog.

**Promotion gate** (default): `F1_drifted(challenger) >= F1_drifted(champion) + min_f1_gain` **and** `F1_original(challenger) >= F1_original(champion) - max_f1_regression`.

In [0]:
def build_eval_set():
  parts = [spark.table(TRAINING_TABLE).sample(fraction=1.0, seed=7).limit(200)
             .select("content", "label").withColumn("segment", F.lit("original"))]
  if spark.catalog.tableExists(INCOMING_TABLE):
    parts.append(spark.table(INCOMING_TABLE).filter("split = 'holdout' AND label IS NOT NULL")
                 .select("content", "label").withColumn("segment", F.lit("drifted")))
  df = parts[0]
  for p in parts[1:]:
    df = df.unionByName(p)
  return df.toPandas()

def evaluate_version(mv, eval_pdf, role):
  pipe = load_pipeline(mv.version)
  preds, latency_ms = predict_contents(pipe, eval_pdf["content"].tolist())
  scored = pd.concat([eval_pdf[["segment", "label"]].reset_index(drop=True), preds], axis=1)
  metrics = {"latency_ms_per_image": latency_ms, "num_parameters": float(pipe.model.num_parameters())}
  for seg, g in [("overall", scored)] + list(scored.groupby("segment")):
    for k, v in clf_metrics(g["label"], g["prediction"], g["p_damaged"]).items():
      metrics[f"{seg}_{k}"] = v
  with mlflow.start_run(run_name=f"eval_v{mv.version}", nested=True) as r:
    mlflow.set_tags({"mlops_stage": "evaluate", "role": role, "model_version": mv.version})
    mlflow.log_metrics(metrics)
    cm = confusion_matrix(scored["label"], scored["prediction"], labels=["normal", "damaged"])
    fig, ax = plt.subplots(figsize=(3.5, 3))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["normal", "damaged"]); ax.set_yticks([0, 1], ["normal", "damaged"])
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title(f"{role} v{mv.version}")
    for i in range(2):
      for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
    mlflow.log_figure(fig, f"confusion_matrix_v{mv.version}.png"); plt.close(fig)
  try:   # MLflow 3: attach the evaluation metrics to the logged model itself
    if getattr(mv, "model_id", None):
      mlflow.log_metrics({f"eval_{k}": v for k, v in metrics.items()}, model_id=mv.model_id)
  except Exception as e:
    print(f"Could not link metrics to logged model: {e}")
  return metrics

if should_run("evaluate"):
  set_task_value("promote", "false")   # default, so the job condition always has a value
  champion = get_version_by_alias(PROD_ALIAS)
  challenger = get_version_by_alias(CHALLENGER_ALIAS)
  if challenger is None or (champion is not None and challenger.version == champion.version):
    print("No new @challenger version to evaluate: comparison skipped")
  else:
    eval_pdf = build_eval_set()
    print(eval_pdf.groupby(["segment", "label"]).size())
    with mlflow.start_run(run_name="champion_vs_challenger") as cmp_run:
      champ_m = evaluate_version(champion, eval_pdf, "champion")
      chall_m = evaluate_version(challenger, eval_pdf, "challenger")
      comparison = pd.DataFrame({"metric": list(champ_m), f"champion_v{champion.version}": list(champ_m.values()),
                                 f"challenger_v{challenger.version}": [chall_m[k] for k in champ_m]})
      comparison["delta"] = comparison.iloc[:, 2] - comparison.iloc[:, 1]

      seg = "drifted" if "drifted_f1" in chall_m else "overall"
      gain = chall_m[f"{seg}_f1"] - champ_m[f"{seg}_f1"]
      regression = champ_m.get("original_f1", 0) - chall_m.get("original_f1", 0)
      promote = gain >= MIN_F1_GAIN and regression <= MAX_F1_REGRESSION
      decision = "promote" if promote else "reject"

      mlflow.set_tags({"dbdemos": "pcb_classification", "mlops_stage": "evaluate", "champion_version": champion.version,
                       "challenger_version": challenger.version, "decision": decision})
      mlflow.log_params({"min_f1_gain": MIN_F1_GAIN, "max_f1_regression": MAX_F1_REGRESSION, "n_eval": len(eval_pdf)})
      mlflow.log_metrics({f"delta_{k}": chall_m[k] - champ_m[k] for k in champ_m})
      mlflow.log_metrics({"f1_gain_drifted": gain, "f1_regression_original": regression, "promote": int(promote)})
      mlflow.log_table(comparison, "champion_vs_challenger.json")

      fig, ax = plt.subplots(figsize=(9, 3.5))
      keys = [k for k in champ_m if k.endswith(("_f1", "_accuracy", "_recall", "_precision"))]
      x = np.arange(len(keys))
      ax.bar(x - 0.2, [champ_m[k] for k in keys], 0.4, label=f"champion v{champion.version}")
      ax.bar(x + 0.2, [chall_m[k] for k in keys], 0.4, label=f"challenger v{challenger.version}")
      ax.set_xticks(x, keys, rotation=45, ha="right"); ax.set_ylim(0, 1.05); ax.legend(); plt.tight_layout()
      mlflow.log_figure(fig, "champion_vs_challenger.png"); plt.show()

    for k, v in {"eval_run_id": cmp_run.info.run_id, "promotion_decision": decision, "f1_gain_drifted": f"{gain:.4f}",
                 "f1_regression_original": f"{regression:.4f}", "compared_with_version": champion.version}.items():
      client.set_model_version_tag(MODEL_NAME, challenger.version, k, v)
    log_event("evaluate", decision, challenger.version, cmp_run.info.run_id, champion_version=champion.version,
              f1_gain_drifted=gain, f1_regression_original=regression)
    set_task_value("promote", str(promote).lower())
    display(comparison)
    print(f"Decision for v{challenger.version}: {decision.upper()} (F1 gain on {seg}: {gain:+.3f}, F1 regression on original: {regression:+.3f})")

## Stage 6 - Redeployment (batch and real time)

Runs only when the latest comparison for the current `@challenger` returned `promote` (and `auto_promote = true`, or when you explicitly run `step = deploy` after a manual review).

**Batch / streaming inference**: the alias swap is the deployment.
1. `@previous` ← current production version (instant rollback target)
2. `@production` ← challenger version
3. `@challenger` is removed

Notebook `03` (and any job using `models:/dbdemos_pcb_classification@production`) picks up the new model on its next run without code changes.

**Real-time inference**: the endpoint needs the base64 pyfunc wrapper built in notebook `03`. The same wrapper is created around the new version and registered as a new version tagged `model_type=rt`, `rt=true` and `source_version=<challenger version>`. Then the endpoint `dbdemos_pcb_classification_endpoint` is updated:
* `rt_rollout = full`: 100% of traffic moves to the new version.
* `rt_rollout = canary`: the current version keeps `100 - canary_pct`% and the new one gets `canary_pct`%. Both are recorded in the existing inference table, so they can be compared on live traffic before you move to 100%.

If the endpoint doesn't exist yet, it is created with the same settings as notebook `03` (CPU, small, scale to zero, inference table enabled).

In [0]:
import base64
from datetime import timedelta
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
from databricks.sdk.service.serving import (ServedEntityInput, ServingModelWorkloadType, TrafficConfig, Route,
                                            AiGatewayConfig, AiGatewayInferenceTableConfig)

class RealtimeCVModelWrapper(mlflow.pyfunc.PythonModel):
  """Same wrapper as notebook 03: decodes base64 images and returns the top class."""
  def __init__(self, pipeline):
    self.pipeline = pipeline
    self.pipeline.model.eval()
  def predict(self, context, images):
    with torch.set_grad_enabled(False):
      images = images["data"].apply(lambda b: Image.open(io.BytesIO(base64.b64decode(b))).convert("RGB")).to_list()
      predictions = self.pipeline(images)
      return pd.DataFrame([max(r, key=lambda x: x["score"]) for r in predictions])

def register_realtime_version(source_version):
  uri = f"models:/{MODEL_NAME}/{source_version}"
  requirements_path = ModelsArtifactRepository(uri).download_artifacts(artifact_path="requirements.txt")
  rt_model = RealtimeCVModelWrapper(mlflow.transformers.load_model(uri, return_type="pipeline", device=-1))  # serving is CPU
  sample = spark.table(TRAINING_TABLE).select("content").limit(2).toPandas()
  df_input = pd.DataFrame({"data": sample["content"].apply(lambda b: base64.b64encode(b).decode("ascii"))})
  with mlflow.start_run(run_name="hugging_face_rt") as run:
    mlflow.set_tags({"dbdemos": "pcb_classification", "rt": "true", "mlops_stage": "deploy", "source_version": source_version})
    info = mlflow.pyfunc.log_model(name="model", python_model=rt_model, pip_requirements=requirements_path,
                                   input_example=df_input,
                                   signature=mlflow.models.infer_signature(df_input, rt_model.predict(None, df_input)))
  rt_mv = mlflow.register_model(info.model_uri, MODEL_NAME)
  for k, v in {"model_type": "rt", "rt": "true", "source_version": str(source_version)}.items():
    client.set_model_version_tag(MODEL_NAME, rt_mv.version, k, v)
  return rt_mv.version

def deploy_to_endpoint(rt_version, rollout="full", canary_pct=10):
  new_entity = ServedEntityInput(name=f"pcb-v{rt_version}", entity_name=MODEL_NAME, entity_version=str(rt_version),
                                 workload_size="Small", workload_type=ServingModelWorkloadType.CPU, scale_to_zero_enabled=True)
  existing = next((e for e in w.serving_endpoints.list() if e.name == SERVING_ENDPOINT_NAME), None)
  if existing is None:
    print(f"Creating endpoint {SERVING_ENDPOINT_NAME} with version {rt_version}...")
    from databricks.sdk.service.serving import EndpointCoreConfigInput
    w.serving_endpoints.create_and_wait(
      name=SERVING_ENDPOINT_NAME, config=EndpointCoreConfigInput(name=SERVING_ENDPOINT_NAME, served_entities=[new_entity]),
      ai_gateway=AiGatewayConfig(inference_table_config=AiGatewayInferenceTableConfig(catalog_name=catalog, schema_name=db, enabled=True)),
      timeout=timedelta(minutes=60))
    return
  current = w.serving_endpoints.get(SERVING_ENDPOINT_NAME).config.served_entities or []
  entities, routes = [new_entity], [Route(served_model_name=new_entity.name, traffic_percentage=100)]
  if rollout == "canary" and current:
    old = current[0]
    entities = [ServedEntityInput(name=old.name, entity_name=old.entity_name, entity_version=old.entity_version,
                                  workload_size=old.workload_size or "Small", workload_type=old.workload_type,
                                  scale_to_zero_enabled=True), new_entity]
    routes = [Route(served_model_name=old.name, traffic_percentage=100 - canary_pct),
              Route(served_model_name=new_entity.name, traffic_percentage=canary_pct)]
  print(f"Updating {SERVING_ENDPOINT_NAME}: " + ", ".join(f"{r.served_model_name}={r.traffic_percentage}%" for r in routes))
  w.serving_endpoints.update_config_and_wait(name=SERVING_ENDPOINT_NAME, served_entities=entities,
                                             traffic_config=TrafficConfig(routes=routes), timeout=timedelta(minutes=60))

if should_run("deploy"):
  challenger = get_version_by_alias(CHALLENGER_ALIAS)
  evaluation = latest_event("evaluate", challenger.version) if challenger else None
  approved = evaluation is not None and evaluation["decision"] == "promote" and (AUTO_PROMOTE or STEP == "deploy")
  if not approved:
    print("Nothing to deploy: no @challenger, not evaluated yet, rejected by the gate, or auto_promote=false")
  else:
    champion = get_version_by_alias(PROD_ALIAS)
    # Batch: alias swap
    if champion is not None:
      client.set_registered_model_alias(MODEL_NAME, PREVIOUS_ALIAS, champion.version)
    client.set_registered_model_alias(MODEL_NAME, PROD_ALIAS, challenger.version)
    client.delete_registered_model_alias(MODEL_NAME, CHALLENGER_ALIAS)
    print(f"@{PROD_ALIAS} -> v{challenger.version} (previous: v{champion.version if champion else None})")
    # Real time: wrapper version + endpoint update
    rt_version = register_realtime_version(challenger.version)
    deploy_to_endpoint(rt_version, RT_ROLLOUT, CANARY_PCT)
    log_event("deploy", f"promoted_{RT_ROLLOUT}", challenger.version, evaluation["mlflow_run_id"],
              previous_version=champion.version if champion else None, rt_version=rt_version, rollout=RT_ROLLOUT)
    print(f"Deployed: batch uses v{challenger.version}, endpoint serves RT v{rt_version} ({RT_ROLLOUT})")

### Rollback

Run the notebook with `step = rollback` to restore the version behind `@previous`: `@production` moves back, and the endpoint is switched to the real-time version built from it (found through the `source_version` tag). If none exists, one is created.

In [0]:
if STEP == "rollback":
  previous = get_version_by_alias(PREVIOUS_ALIAS)
  assert previous is not None, f"No @{PREVIOUS_ALIAS} version to roll back to"
  current = get_version_by_alias(PROD_ALIAS)
  client.set_registered_model_alias(MODEL_NAME, PROD_ALIAS, previous.version)
  rt_candidates = [v.version for v in client.search_model_versions(f"name='{MODEL_NAME}'")
                   if client.get_model_version(MODEL_NAME, v.version).tags.get("source_version") == str(previous.version)]
  rt_version = max(rt_candidates, key=int) if rt_candidates else register_realtime_version(previous.version)
  deploy_to_endpoint(rt_version, "full")
  log_event("rollback", "rolled_back", previous.version, None, from_version=current.version if current else None, rt_version=rt_version)
  print(f"@{PROD_ALIAS} -> v{previous.version}, endpoint -> RT v{rt_version}")

## Stage 7 - Automate the loop with a Lakeflow Job

The cell below creates (or updates) a job named `dbdemos_pcb_mlops_drift_retraining` that runs **this notebook** once per stage, passing a different `step` parameter:

```
drift_check --> [drift_detected == true?] --> retrain --> evaluate --> [promote == true?] --> deploy
```

* The two diamonds are **condition tasks** that read the task values published by the notebook (`drift_detected`, `promote`). When a condition is false, the following tasks are skipped and the run still succeeds.
* The job runs on serverless compute, like the existing demo job. For faster retraining, select a GPU in the notebook's Environment panel (the retraining recipe automatically uses the larger ViT model on GPU).
* A daily schedule is configured **paused** by default; unpause it in the Jobs UI when you're ready. You can also replace the schedule with a *table update* trigger on the table where new images land.
* The existing job `dbdemos_computer_vision_pcb_ramin_amiri` (notebooks 01-04) is not modified.

The job is only created when the notebook runs interactively with `step = all` and `create_job = true`.

In [0]:
JOB_NAME = "dbdemos_pcb_mlops_drift_retraining"

def running_in_job():
  try:
    return dbutils.notebook.entry_point.getDbutils().notebook().getContext().jobId().isDefined()
  except Exception:
    return False

if STEP == "all" and CREATE_JOB and not running_in_job():
  notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
  common = {k: dbutils.widgets.get(k) for k in ["simulate_drift", "include_endpoint_traffic", "psi_threshold", "ks_alpha",
            "min_drifted_features", "confidence_drop", "f1_drop", "num_train_epochs", "min_f1_gain", "max_f1_regression",
            "auto_promote", "rt_rollout", "canary_pct", "enable_monitoring"]}
  def nb_task(key, step, depends_on=None, outcome=None):
    t = {"task_key": key, "environment_key": "env_default",
         "notebook_task": {"notebook_path": notebook_path, "source": "WORKSPACE",
                           "base_parameters": {**common, "step": step, "create_job": "false", "force_retrain": "false"}}}
    if depends_on:
      t["depends_on"] = [{"task_key": depends_on, **({"outcome": outcome} if outcome else {})}]
    return t
  settings = {
    "name": JOB_NAME,
    "environments": [{"environment_key": "env_default", "spec": {"client": "5"}}],
    "max_concurrent_runs": 1,
    "schedule": {"quartz_cron_expression": "0 0 6 * * ?", "timezone_id": "UTC", "pause_status": "PAUSED"},
    "tags": {"dbdemos": "pcb_classification", "mlops": "drift_retraining"},
    "tasks": [
      nb_task("drift_check", "drift_check"),
      {"task_key": "is_drift_detected", "depends_on": [{"task_key": "drift_check"}],
       "condition_task": {"op": "EQUAL_TO", "left": "{{tasks.drift_check.values.drift_detected}}", "right": "true"}},
      nb_task("retrain", "retrain", "is_drift_detected", "true"),
      nb_task("evaluate", "evaluate", "retrain"),
      {"task_key": "is_promotion_approved", "depends_on": [{"task_key": "evaluate"}],
       "condition_task": {"op": "EQUAL_TO", "left": "{{tasks.evaluate.values.promote}}", "right": "true"}},
      nb_task("deploy", "deploy", "is_promotion_approved", "true"),
    ]}
  existing_job = next(iter(w.jobs.list(name=JOB_NAME)), None)
  if existing_job:
    w.api_client.do("POST", "/api/2.2/jobs/reset", body={"job_id": existing_job.job_id, "new_settings": settings})
    job_id = existing_job.job_id
    print(f"Updated job {JOB_NAME} ({job_id})")
  else:
    job_id = w.api_client.do("POST", "/api/2.2/jobs/create", body=settings)["job_id"]
    print(f"Created job {JOB_NAME} ({job_id})")
  displayHTML(f'<a href="/#job/{job_id}" target="_blank">Open the MLOps job {JOB_NAME}</a> (schedule paused - unpause it to automate the loop)')

## Audit trail: decisions and model versions

Every decision (drift / no drift, registered challenger, promote / reject, deployed, rolled back) is appended to `pcb_mlops_events` with the MLflow run that justifies it. The registry view shows each version, its aliases and its MLOps tags.

In [0]:
if spark.catalog.tableExists(EVENTS_TABLE):
  display(spark.table(EVENTS_TABLE).orderBy(F.col("event_ts").desc()).limit(20))

aliases = {}
for a, v in client.get_registered_model(MODEL_NAME).aliases.items():
  aliases.setdefault(str(v), []).append(a)
rows = []
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
  mv = client.get_model_version(MODEL_NAME, v.version)
  rows.append({"version": int(mv.version), "aliases": ", ".join(aliases.get(str(mv.version), [])),
               "model_type": mv.tags.get("model_type", ""), "trigger": mv.tags.get("trigger", ""),
               "source_version": mv.tags.get("source_version", ""), "promotion_decision": mv.tags.get("promotion_decision", ""),
               "f1_gain_drifted": mv.tags.get("f1_gain_drifted", ""), "run_id": mv.run_id})
display(pd.DataFrame(rows).sort_values("version", ascending=False))

# How the whole process works

## 1. The big picture

```
             +----------------------------- existing demo (unchanged) ------------------------------+
             | 01 ingestion/ETL -> 02 training -> @production -> 03 batch + real-time -> 04 / 05     |
             +-------------------------------------^-----------------------------+------------------+
                                                   | alias swap                  | requests logged
 new images                                        |                             v
 (cameras, endpoint) -> pcb_incoming_images -> [drift_check] -> [retrain] -> [evaluate] -> [deploy]
                                                    |              |            |            |
                                                    v              v            v            v
                                    pcb_inference_log      new UC version   MLflow      @production,
                                    + MLflow drift_check   @challenger      comparison  @previous,
                                    + DQ monitor                            + tags      endpoint update

                        every decision -> pcb_mlops_events (audit trail)
```

The production model is always identified by the Unity Catalog alias `@production` on `ramin_serverless_aws_catalog.symcor.dbdemos_pcb_classification`. Every consumer (notebook `03`, the batch UDF, any downstream job) resolves that alias when it loads the model. **Changing the alias is the deployment**, which is why the existing notebooks don't need any change.

## 2. Step by step

**Stage 1 - Incoming data.** New images are appended to `pcb_incoming_images` with a `batch_id`, an optional `label` and a deterministic `train`/`holdout` split. In this demo, a batch with realistic capture changes is simulated; in production an ingestion job (for example Auto Loader as in notebook `01`) writes there, and the endpoint inference table can be added as a second source.

**Stage 2 - Score and profile.** The `@production` model scores a fixed reference sample of the training data (`pcb_drift_baseline`) and the new batch (`pcb_inference_log`). For each image we store explainable statistics (brightness, contrast, sharpness, entropy, colour means) plus the prediction, its confidence and `P(damaged)`.

**Stage 3 - Drift detection.** Baseline and current batch are compared:
* *Data drift*: PSI (how far the distribution moved) **and** a KS test (whether the move is statistically significant) per feature. Both must agree, which avoids false alarms on small batches.
* *Prediction drift*: shift in the predicted-label mix and drop in mean confidence. Useful even without labels.
* *Performance drift*: F1 drop when labels are available, the most direct signal.

The output is an MLflow run `drift_check` (metrics, report table, histograms, dataset lineage, decision tag), an event in `pcb_mlops_events`, and the task value `drift_detected`. In parallel, the **Data Quality Monitoring** inference monitor on `pcb_inference_log` keeps long-term drift and quality metrics per day, per model version and per source, with an auto-generated dashboard.

**Stage 4 - Retraining.** Triggered only by drift (or `force_retrain`). The model is fine-tuned with the same recipe as notebook `02` on the original data plus the newly labelled images, logged in the run `hugging_face_retrain` (linked to the drift run that triggered it), registered as a **new version** and given the `@challenger` alias. Nothing is overwritten: previous runs, models and versions all remain in MLflow and Unity Catalog.

**Stage 5 - Comparison.** `@challenger` and `@production` are evaluated on the same held-out data, split into `original` and `drifted` segments. The parent run `champion_vs_challenger` holds the side-by-side table, deltas, chart and decision; nested runs `eval_v<N>` hold each model's metrics and confusion matrix (select them in the MLflow UI and click **Compare**). The gate promotes the challenger only if it's better on the new conditions and not meaningfully worse on the original ones. The decision is also written as tags on the challenger model version, visible in Catalog Explorer.

**Stage 6 - Redeployment.** If approved:
* batch: `@previous` ← old production, `@production` ← challenger, `@challenger` removed;
* real time: the base64 wrapper from notebook `03` is rebuilt around the new version, registered as a new version tagged `model_type=rt` / `source_version`, and the serving endpoint is updated (full switch or canary split). Endpoint traffic keeps being logged to the inference table, so the next drift check can include it.

`step = rollback` restores `@previous` on both the alias and the endpoint in one run.

**Stage 7 - Automation.** The Lakeflow Job `dbdemos_pcb_mlops_drift_retraining` runs this notebook as four notebook tasks with two condition tasks in between. State moves between tasks through **task values** (for the conditions) and through the **Unity Catalog registry and `pcb_mlops_events`** (for the details), so each task can run on its own compute and be retried on its own.

## 3. Where to look

| What | Where |
|---|---|
| Drift metrics, histograms, decision | MLflow experiment `/Shared/dbdemos/experiments/computer-vision-dl/pcb`, runs `drift_check` |
| Retraining | runs `hugging_face_retrain` (training metrics, logged model, dataset inputs) |
| New vs previous version | runs `champion_vs_challenger` and nested `eval_v<N>` |
| Versions, aliases, decision tags, lineage | Catalog Explorer > `dbdemos_pcb_classification` |
| Long-term drift and quality | `pcb_inference_log_drift_metrics`, `pcb_inference_log_profile_metrics`, monitor dashboard |
| Live traffic | `dbdemos_pcb_classification_endpoint_payload` |
| Decision history | `pcb_mlops_events` |

## 4. Compatibility with notebooks 01-05

* No existing notebook, config or resource file is modified. The new tables have their own names.
* `@production` always points to a **transformers** version (never to an RT wrapper version), so notebook `03` keeps loading it with `mlflow.transformers.load_model` and building its own wrapper.
* Notebook `03`'s endpoint cell does nothing when the endpoint already exists (`force_update = False`), so it won't undo a deployment made here.
* Caveat: rerunning notebook `02` by hand still moves `@production` immediately (its original behaviour). To keep all promotions behind the gate, retrain from this notebook (`step = retrain` or `force_retrain = true`).

## 5. Taking it further

* Replace the simulated batch with real labelled images from QA, and set `simulate_drift = false` in the job.
* Replace the paused schedule with a table update trigger on the landing table, or add a SQL alert on the monitor's drift metrics table.
* Add a manual approval: set `auto_promote = false` and run `step = deploy` after review, or attach an MLflow 3 deployment job to the registered model.
* Use a frozen, labelled holdout table for the `original` segment of the comparison.
* Run the `retrain` task on serverless GPU to use the larger ViT model.